### The ra/dec coordinates from the Bersier & Wood dataset sometimes were less accurate than the coordinates in their names
### This notebook aims to solve that problem by replacing the ra/dec values with the values in each object's name

In [2]:
#Imports
from astroquery.gaia import Gaia
from astroquery.vizier import Vizier
from astropy.coordinates import SkyCoord
import astropy.units as units
import csv
import pandas as pd
from lsst.rsp import get_tap_service

from sklearn import svm

In [3]:
Vizier.ROW_LIMIT = -1

In [4]:
catalog_list = Vizier.get_catalogs('J/AJ/123/840')

In [5]:
print(catalog_list.keys())

['J/AJ/123/840/table1', 'J/AJ/123/840/table2', 'J/AJ/123/840/table3', 'J/AJ/123/840/table4']


**Table 1 contains information about the rest of the tables**

**Table 2, 3, and 4 are rrl, ceph, lpv**

In [6]:
bw_table_rrl = catalog_list[1]
bw_table_ceph = catalog_list[2]
bw_table_lpv = catalog_list[3]

In [7]:
bw_df_rrl = bw_table_rrl.to_pandas()
bw_df_ceph = bw_table_ceph.to_pandas()
bw_df_lpv = bw_table_lpv.to_pandas()

**Adding labels so that objects can be identified after merging tables**

In [8]:
bw_df_rrl['Main Class'] = 'RR Lyrae'
bw_df_ceph['Main Class'] = 'Cepheid'
bw_df_lpv['Main Class'] = 'Long Period Variable'

In [9]:
bw_df_all = pd.concat([bw_df_rrl, bw_df_ceph, bw_df_lpv], axis=0, ignore_index=False)

In [10]:
bw_df_all

,FBW,o_Vmag,Vmag,e_Vmag,o_Icmag,Icmag,e_Icmag,WS,Period,E(B-V),_RA,_DE,Main Class,V-Ic,e_V-Ic,Per,Class,o_V-Ic,OName,n_FBW
0,J023926.4-344027,28.0,21.320999,0.028,27.0,20.799000,0.042,1.030,0.37206,0.025,39.8600,-34.6742,RR Lyrae,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,J023749.8-342427,24.0,21.389000,0.027,24.0,20.754999,0.036,1.422,0.54365,0.028,39.4575,-34.4075,RR Lyrae,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J023750.8-342736,32.0,21.346001,0.021,31.0,20.743999,0.035,2.741,0.58252,0.030,39.4617,-34.4600,RR Lyrae,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J023751.9-342620,32.0,21.419001,0.023,32.0,20.886999,0.033,1.139,0.58052,0.029,39.4662,-34.4389,RR Lyrae,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J023752.8-344615,32.0,21.469000,0.029,32.0,20.559999,0.033,2.005,0.16459,0.034,39.4700,-34.7708,RR Lyrae,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,J024048.3-343536,14.0,17.375000,0.013,NaN,NaN,NaN,NaN,NaN,0.023,40.2012,-34.5933,Long Period Variable,1.922,0.021,NaN,NaN,11.0,S144,?
81,J024052.2-343723,15.0,18.370001,0.015,NaN,NaN,NaN,NaN,NaN,0.023,40.2175,-34.6231,Long Period Variable,2.100,0.025,NaN,NaN,14.0,DK23,?
82,J024103.5-344805,33.0,20.412001,0.097,NaN,NaN,NaN,NaN,NaN,0.024,40.2646,-34.8014,Long Period Variable,3.793,0.101,NaN,NaN,34.0,,
83,J024110.8-343151,14.0,18.513000,0.015,NaN,NaN,NaN,NaN,NaN,0.022,40.2950,-34.5308,Long Period Variable,2.107,0.025,NaN,NaN,13.0,"S153, DI16",


In [11]:
bw_df_all['Class'].unique()

array([nan, 'P2C', 'AC', 'blue', 'P2C?'], dtype=object)

**New subset with only FBW (name), _RA, and _DE (coordinates)**

In [12]:
bw_df_all_subset = bw_df_all[['FBW', '_RA', '_DE']]

**Making another subsetwith only name**

In [13]:
bw_df_names = bw_df_all_subset['FBW']

In [15]:
bw_df_names

0     J023926.4-344027
1     J023749.8-342427
2     J023750.8-342736
3     J023751.9-342620
4     J023752.8-344615
            ...       
80    J024048.3-343536
81    J024052.2-343723
82    J024103.5-344805
83    J024110.8-343151
84    J024111.5-345504
Name: FBW, Length: 624, dtype: object

**Converting names to coordinates**

In [17]:
bw_df_coords = SkyCoord(bw_df_names.tolist(), unit=(units.hourangle, units.deg), frame='icrs')

In [18]:
bw_df_ra = bw_df_coords.ra.deg
bw_df_dec = bw_df_coords.dec.deg

In [19]:
bw_df_ra

array([39.86      , 39.4575    , 39.46166667, 39.46625   , 39.47      ,
       39.4725    , 39.47916667, 39.48333333, 39.48375   , 39.48583333,
       39.4875    , 39.4875    , 39.49      , 39.49375   , 39.495     ,
       39.49625   , 39.49833333, 39.49833333, 39.50041667, 39.50041667,
       39.50125   , 39.50166667, 39.50458333, 39.50583333, 39.50666667,
       39.50875   , 39.51166667, 39.51791667, 39.51916667, 39.51958333,
       39.52708333, 39.52708333, 39.52875   , 39.53      , 39.53      ,
       39.53458333, 39.53666667, 39.54208333, 39.54458333, 39.54583333,
       39.55375   , 39.55375   , 39.55625   , 39.56791667, 39.57      ,
       39.575     , 39.57625   , 39.58125   , 39.58166667, 39.58291667,
       39.58291667, 39.58583333, 39.59      , 39.59041667, 39.5925    ,
       39.59916667, 39.6025    , 39.60625   , 39.61125   , 39.6125    ,
       39.61458333, 39.61666667, 39.61666667, 39.61916667, 39.61916667,
       39.62      , 39.62291667, 39.625     , 39.62541667, 39.63

In [20]:
bw_df_all_subset.loc[:, 'New RA'] = bw_df_ra
bw_df_all_subset.loc[:, 'New DEC'] = bw_df_dec

/tmp/ipykernel_17617/2021585380.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bw_df_all_subset.loc[:, 'New RA'] = bw_df_ra
/tmp/ipykernel_17617/2021585380.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bw_df_all_subset.loc[:, 'New DEC'] = bw_df_dec


In [21]:
bw_df_all_subset

,FBW,_RA,_DE,New RA,New DEC
0,J023926.4-344027,39.8600,-34.6742,39.860000,-34.674167
1,J023749.8-342427,39.4575,-34.4075,39.457500,-34.407500
2,J023750.8-342736,39.4617,-34.4600,39.461667,-34.460000
3,J023751.9-342620,39.4662,-34.4389,39.466250,-34.438889
4,J023752.8-344615,39.4700,-34.7708,39.470000,-34.770833
...,...,...,...,...,...
80,J024048.3-343536,40.2012,-34.5933,40.201250,-34.593333
81,J024052.2-343723,40.2175,-34.6231,40.217500,-34.623056
82,J024103.5-344805,40.2646,-34.8014,40.264583,-34.801389
83,J024110.8-343151,40.2950,-34.5308,40.295000,-34.530833


In [22]:
path = os.getcwd()
vstars_path = os.path.join(path, 'v_stars.csv')
vstars_path

'/home/jakedlr/notebooks/tutorials/v_stars.csv'

In [23]:
v_stars_df = pd.read_csv(vstars_path)

In [24]:
v_stars_df

,ra,dec,period,Vmag,e_Vmag,Icmag,e_Icmag,E(B-V),source,main category,...,iamp,e_iamp,vamp_sec,e_vamp_sec,bamp_sec,e_bamp_sec,gmag,bpmag,rpmag,bp-rp
0,39.860000,-34.674200,0.372060,21.321,0.028,20.799,0.042,0.025,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,39.457500,-34.407500,0.543650,21.389,0.027,20.755,0.036,0.028,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,39.461700,-34.460000,0.582520,21.346,0.021,20.744,0.035,0.030,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,39.466200,-34.438900,0.580520,21.419,0.023,20.887,0.033,0.029,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,39.470000,-34.770800,0.164590,21.469,0.029,20.560,0.033,0.034,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
894,40.441648,-34.291025,NaN,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,17.425013,18.622169,16.353251,2.268919
895,40.222196,-34.203659,243.829816,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,17.899149,18.981697,16.645916,2.335781
896,39.742913,-34.536605,203.290829,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,17.573246,18.800655,16.482256,2.318399
897,39.919195,-34.337461,163.279808,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,16.944231,18.394299,15.806978,2.587320


**Setting ra and dec in v_stars_df to the newly calculated ra/dec**

In [25]:
v_stars_df.loc[v_stars_df['source'] == 'Bersier Wood', 'ra'] = bw_df_ra
v_stars_df.loc[v_stars_df['source'] == 'Bersier Wood', 'dec'] = bw_df_dec

In [29]:
v_stars_df

,ra,dec,period,Vmag,e_Vmag,Icmag,e_Icmag,E(B-V),source,main category,...,iamp,e_iamp,vamp_sec,e_vamp_sec,bamp_sec,e_bamp_sec,gmag,bpmag,rpmag,bp-rp
0,39.860000,-34.674167,0.372060,21.321,0.028,20.799,0.042,0.025,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,39.457500,-34.407500,0.543650,21.389,0.027,20.755,0.036,0.028,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,39.461667,-34.460000,0.582520,21.346,0.021,20.744,0.035,0.030,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,39.466250,-34.438889,0.580520,21.419,0.023,20.887,0.033,0.029,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,39.470000,-34.770833,0.164590,21.469,0.029,20.560,0.033,0.034,Bersier Wood,RRLyrae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
894,40.441648,-34.291025,NaN,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,17.425013,18.622169,16.353251,2.268919
895,40.222196,-34.203659,243.829816,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,17.899149,18.981697,16.645916,2.335781
896,39.742913,-34.536605,203.290829,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,17.573246,18.800655,16.482256,2.318399
897,39.919195,-34.337461,163.279808,NaN,NaN,NaN,NaN,NaN,Gaia,LPV,...,NaN,NaN,NaN,NaN,NaN,NaN,16.944231,18.394299,15.806978,2.587320


**Move to csv**

In [72]:
v_stars_df.to_csv('v_stars_updated.csv', index=False)